# Load NHTSA complaints into Pinecone

Seeds the `carverdict` index with real complaint data for a set of popular vehicles, so the chat has something to search out of the box. Run this once to (re)load the index.

In [1]:
import os, json, time, warnings
import requests
from dotenv import load_dotenv
from pinecone import Pinecone

warnings.filterwarnings("ignore")
load_dotenv(".env.local")

GEMINI_KEY = os.environ["GEMINI_API_KEY"]
PINECONE_KEY = os.environ["PINECONE_API_KEY"]
INDEX_NAME = os.environ["PINECONE_INDEX"]

/Users/sufiyanrauf/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Popular vehicles to seed. NHTSA's complaints API has its own model vocabulary,
# so some entries use the exact string it expects (e.g. the F-150 is filed under body
# styles like "F-150 SUPERCAB"). We keep the clean name but query the string NHTSA wants.
vehicles = [
    ("Toyota", "Camry", 2019, "Camry"),
    ("Toyota", "Corolla", 2019, "Corolla"),
    ("Toyota", "RAV4", 2019, "RAV4"),
    ("Toyota", "Highlander", 2020, "Highlander"),
    ("Toyota", "Tacoma", 2019, "Tacoma"),
    ("Honda", "Accord", 2019, "Accord"),
    ("Honda", "Civic", 2019, "Civic"),
    ("Honda", "CR-V", 2019, "CR-V"),
    ("Honda", "Pilot", 2019, "Pilot"),
    ("Honda", "Odyssey", 2019, "Odyssey"),
    ("Ford", "Explorer", 2020, "Explorer"),
    ("Ford", "Escape", 2020, "Escape"),
    ("Ford", "Mustang", 2019, "Mustang"),
    ("Ford", "F-150", 2020, "F-150 SUPERCAB"),
    ("Chevrolet", "Silverado 1500", 2020, "Silverado 1500"),
    ("Chevrolet", "Equinox", 2019, "Equinox"),
    ("Chevrolet", "Malibu", 2019, "Malibu"),
    ("Tesla", "Model 3", 2021, "Model 3"),
    ("Tesla", "Model Y", 2020, "Model Y"),
    ("Nissan", "Altima", 2019, "Altima"),
    ("Nissan", "Rogue", 2019, "Rogue"),
    ("Jeep", "Grand Cherokee", 2019, "Grand Cherokee"),
    ("Jeep", "Wrangler", 2019, "Wrangler"),
    ("Hyundai", "Elantra", 2019, "Elantra"),
    ("Subaru", "Outback", 2019, "Outback"),
    ("Ram", "1500", 2019, "1500"),
    ("GMC", "Sierra 1500", 2019, "Sierra 1500"),
]
per_vehicle = 6

def get_complaints(make, query_model, year):
    url = "https://api.nhtsa.gov/complaints/complaintsByVehicle"
    r = requests.get(url, params={"make": make, "model": query_model, "modelYear": year}, timeout=30)
    r.raise_for_status()
    return r.json().get("results", [])

records = []
for make, model, year, query_model in vehicles:
    results = get_complaints(make, query_model, year)
    print(f"{make} {model} {year}: {len(results)} complaints found")
    for c in results[:per_vehicle]:
        summary = (c.get("summary") or "").strip()
        if not summary:
            continue
        records.append({
            "id": str(c["odiNumber"]),
            "make": make,
            "model": model,
            "year": year,
            "components": [x.strip() for x in (c.get("components") or "").split(",") if x.strip()],
            "date": c.get("dateComplaintFiled", ""),
            "crash": bool(c.get("crash")),
            "fire": bool(c.get("fire")),
            "injuries": c.get("numberOfInjuries", 0),
            "deaths": c.get("numberOfDeaths", 0),
            "summary": summary,
        })

print("total records:", len(records))

Toyota Camry 2019: 379 complaints found
Toyota Corolla 2019: 200 complaints found


Toyota RAV4 2019: 866 complaints found
Toyota Highlander 2020: 292 complaints found


Toyota Tacoma 2019: 209 complaints found
Honda Accord 2019: 637 complaints found


Honda Civic 2019: 348 complaints found
Honda CR-V 2019: 1056 complaints found


Honda Pilot 2019: 849 complaints found
Honda Odyssey 2019: 866 complaints found


...

Tesla Model Y 2020: 271 complaints found
Nissan Altima 2019: 221 complaints found


Nissan Rogue 2019: 329 complaints found
Jeep Grand Cherokee 2019: 354 complaints found


Jeep Wrangler 2019: 717 complaints found
Hyundai Elantra 2019: 239 complaints found


Subaru Outback 2019: 1027 complaints found
Ram 1500 2019: 1440 complaints found


GMC Sierra 1500 2019: 598 complaints found
total records: 162


In [3]:
# Gemini embeddings, 768 dims to match the Pinecone index.
# Key goes in a header (not the URL) so it never ends up in logs or errors.
EMBED_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents"
HEADERS = {"x-goog-api-key": GEMINI_KEY, "Content-Type": "application/json"}

def embed_batch(texts):
    body = {"requests": [
        {"model": "models/gemini-embedding-001",
         "content": {"parts": [{"text": t}]},
         "outputDimensionality": 768}
        for t in texts
    ]}
    for attempt in range(5):
        r = requests.post(EMBED_URL, headers=HEADERS, json=body, timeout=120)
        if r.status_code == 429:   # free tier rate limit, wait and retry
            time.sleep(15 * (attempt + 1))
            continue
        r.raise_for_status()
        return [e["values"] for e in r.json()["embeddings"]]
    raise RuntimeError("embedding failed after retries")

vectors = []
batch = 20
for i in range(0, len(records), batch):
    chunk = records[i:i+batch]
    embs = embed_batch([r["summary"] for r in chunk])
    for r, emb in zip(chunk, embs):
        meta = {k: r[k] for k in ("make","model","year","components","date","crash","fire","injuries","deaths","summary")}
        vectors.append({"id": r["id"], "values": emb, "metadata": meta})
    print(f"embedded {len(vectors)}/{len(records)}")
    time.sleep(10)

embedded 20/162


embedded 40/162


embedded 60/162


embedded 80/162


embedded 100/162


embedded 120/162


embedded 140/162


embedded 160/162


embedded 162/162


In [4]:
pc = Pinecone(api_key=PINECONE_KEY)
index = pc.Index(INDEX_NAME)
index.delete(delete_all=True)   # clear so re-running gives a clean index
index.upsert(vectors=vectors)
time.sleep(2)
print("upserted", len(vectors), "vectors into", INDEX_NAME)
print(index.describe_index_stats())

upserted 162 vectors into carverdict
{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 162}},
 'total_vector_count': 162,
 'vector_type': 'dense'}


In [5]:
with open("client-complaints.json", "w") as f:
    json.dump(records, f, indent=2)
print("saved", len(records), "records to client-complaints.json")

saved 162 records to client-complaints.json
